# ES Audit — Public & Private Collection Demo

Demonstrates two approaches on the live review environment:
- **Approach A** — Public collection: items indexed to the shared `dynastore-items` ES alias, searchable without authentication
- **Approach B** — Private collection: items go to the per-catalog ES only, STAC search returns nothing for anonymous callers

Covers fixes from PR #408:
| Fix | Verified by |
|-----|-------------|
| Items SEARCH routing ES→PG | Approach A step 4 |
| OUTBOX alias maintenance | Approach A step 5 |
| `is_private` ES field | Approach B step 3 |
| Privacy apply-handler sync | Approach B step 3 |
| STAC error surfacing | Approach B step 6 |
| Registry re-key | step 0 (config resolve) |

**Prerequisites:** set `DYNASTORE_TOKEN` env var to a sysadmin Bearer token from the review env.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "https://data.review.fao.org/geospatial/v2/api/catalog")
TOKEN   = os.environ.get("DYNASTORE_TOKEN", "")
RUN_ID  = os.environ.get("RUN_ID", uuid.uuid4().hex[:6])

PUB_CATALOG_ID  = f"audit-pub-{RUN_ID}"
PRIV_CATALOG_ID = f"audit-priv-{RUN_ID}"
PUB_COLL_ID     = f"pub-coll-{RUN_ID}"
PRIV_COLL_ID    = f"priv-coll-{RUN_ID}"

client = httpx.Client(base_url=BASE_URL, headers={"Authorization": f"Bearer {TOKEN}"}, timeout=60.0)

print(f"Base URL : {BASE_URL}")
print(f"Run ID   : {RUN_ID}")
print(f"Token    : {'set ✓' if TOKEN else '*** MISSING — set DYNASTORE_TOKEN ***'}")


## Step 0 — Verify registry re-key fix

GET `/configs/.../plugins/items_postgresql_driver` must return 200 (was 404 before the fix).

In [ ]:
# We need a catalog first to probe config resolution.
# Use the public catalog created in Approach A — but we create it now.
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PUB_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"ES Audit public {RUN_ID}",
    "description": "Ephemeral — safe to delete",
    "links": [],
}))
assert r.status_code in (200, 201, 409), f"Catalog create failed: {r.status_code} {r.text[:200]}"
print(f"Catalog {PUB_CATALOG_ID!r}: {r.status_code}")

r = client.get(f"/configs/catalogs/{PUB_CATALOG_ID}/plugins/items_postgresql_driver")
print(f"GET items_postgresql_driver config: {r.status_code}")
if r.status_code == 200:
    print("  ✓  registry re-key fix ACTIVE")
elif r.status_code == 404:
    print("  ✗  registry re-key fix NOT active (PR #408 not yet deployed)")
else:
    print(f"  ?  unexpected {r.status_code}: {r.text[:150]}")


## Approach A — Public collection

Items ingest → OUTBOX drain → `dynastore-items` alias → anonymous search succeeds.

### A.1 — Create public collection

In [ ]:
r = client.post(f"/stac/catalogs/{PUB_CATALOG_ID}/collections", content=json.dumps({
    "id": PUB_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Public items demo",
    "links": [], "title": "Public Items",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "license": "proprietary",
}))
assert r.status_code in (200, 201, 409), f"Collection create failed: {r.status_code} {r.text[:200]}"
print(f"Collection {PUB_COLL_ID!r}: {r.status_code}")


### A.2 — Ingest 3 public STAC items

In [ ]:
PUB_ITEM_IDS = []
for i in range(3):
    item_id = f"pub-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.1, 41.9]},
            "bbox": [12.4 + i * 0.1, 41.8, 12.6 + i * 0.1, 42.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z", "run": RUN_ID},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PUB_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ✓")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:120]}")

print(f"\nIndexed {len(PUB_ITEM_IDS)}/3 items. Waiting 10 s for OUTBOX drain …")
time.sleep(10)


### A.3 — Verify HATEOAS links use PUT not PATCH

In [ ]:
r = client.get(f"/configs/catalogs/{PUB_CATALOG_ID}/plugins/items_elasticsearch_driver")
print(f"GET items_elasticsearch_driver config: {r.status_code}")
if r.status_code == 200:
    links = r.json().get("_links", [])
    for lk in links:
        rel = lk.get("rel")
        method = lk.get("method", "?").upper()
        href = lk.get("href", "")[-60:]
        flag = "✗ STALE" if method == "PATCH" else "✓"
        print(f"  {flag} rel={rel!r:20} method={method:6} ...{href}")
else:
    print(f"  (driver not registered on this catalog — skip)")


### A.4 — STAC search returns items (ES routing fix)

In [ ]:
r = client.post(
    f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
)
print(f"POST /search/…/items-search: {r.status_code}")
if r.ok:
    body = r.json()
    feats = body.get("features", [])
    total = body.get("features_count", body.get("numberMatched", "?"))
    print(f"  features returned: {len(feats)} (total≈{total})")
    for f in feats[:3]:
        print(f"    {f['id']}")
    if len(feats) > 0:
        print("  ✓  ES SEARCH routing fix ACTIVE")
    else:
        print("  ✗  No items — either routing fix not deployed or OUTBOX not drained yet")
        print(f"  full body: {json.dumps(body)[:400]}")
else:
    print(f"  error: {r.text[:300]}")


### A.5 — Anonymous search also returns items (public collection)

In [ ]:
anon = httpx.Client(base_url=BASE_URL, timeout=60.0)
r = anon.post(
    f"/search/catalogs/{PUB_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
    headers={"Content-Type": "application/json"},
)
print(f"anonymous POST /search/…/items-search: {r.status_code}")
feats = r.json().get("features", []) if r.ok else []
print(f"  features: {len(feats)}")
if len(feats) > 0:
    print("  ✓  public items visible without auth")
else:
    print("  ✗  no items for anonymous — check routing + OUTBOX drain")
anon.close()


---
## Approach B — Private collection

Items marked `is_private=true` → ES doc gets `is_private` field → anonymous SEARCH is blocked.

### B.1 — Create private catalog + collection

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PRIV_CATALOG_ID,
    "type": "Catalog", "stac_version": "1.0.0",
    "title": f"ES Audit private {RUN_ID}",
    "description": "Ephemeral — safe to delete",
    "links": [],
}))
assert r.status_code in (200, 201, 409), f"Catalog create failed: {r.status_code} {r.text[:200]}"
print(f"Catalog {PRIV_CATALOG_ID!r}: {r.status_code}")

r = client.post(f"/stac/catalogs/{PRIV_CATALOG_ID}/collections", content=json.dumps({
    "id": PRIV_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Private items demo",
    "links": [], "title": "Private Items",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "license": "proprietary",
}))
assert r.status_code in (200, 201, 409), f"Collection create failed: {r.status_code} {r.text[:200]}"
print(f"Collection {PRIV_COLL_ID!r}: {r.status_code}")


### B.2 — Mark the collection private

In [ ]:
r = client.put(
    f"/configs/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/plugins/collection_privacy",
    content=json.dumps({"is_private": True}),
)
print(f"PUT collection_privacy: {r.status_code}")
if r.ok:
    print(f"  config: {r.json()}")
else:
    print(f"  body: {r.text[:200]}")


### B.3 — Ingest 3 private items

In [ ]:
PRIV_ITEM_IDS = []
for i in range(3):
    item_id = f"priv-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0",
            "id": item_id,
            "geometry": {"type": "Point", "coordinates": [13.5 + i * 0.1, 42.9]},
            "bbox": [13.4 + i * 0.1, 42.8, 13.6 + i * 0.1, 43.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z", "confidential": True},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PRIV_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code} ✓")
    elif r.status_code == 500:
        print(f"  item {item_id}: 500 — check if STAC error surfacing fix is deployed")
        try:
            print(f"    detail: {r.json()}")
        except Exception:
            print(f"    raw: {r.text[:200]}")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:120]}")

print(f"\nWaiting 10 s for OUTBOX drain …")
time.sleep(10)


### B.4 — Authenticated search returns private items

In [ ]:
r = client.post(
    f"/search/catalogs/{PRIV_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
)
print(f"authenticated POST /search/…/items-search: {r.status_code}")
if r.ok:
    feats = r.json().get("features", [])
    print(f"  features: {len(feats)}")
    if feats:
        print("  ✓  private items visible to authenticated user")
    else:
        print("  ✗  0 features — routing may not be resolving private ES driver")
else:
    print(f"  error: {r.text[:300]}")


### B.5 — Anonymous search is blocked

In [ ]:
anon = httpx.Client(base_url=BASE_URL, timeout=60.0)
r = anon.post(
    f"/search/catalogs/{PRIV_CATALOG_ID}/items-search",
    content=json.dumps({"limit": 10}),
    headers={"Content-Type": "application/json"},
)
print(f"anonymous POST /search/…/items-search: {r.status_code}")
body = r.json() if r.headers.get("content-type","").startswith("application/json") else {}
feats = body.get("features", [])
if r.status_code in (401, 403):
    print("  ✓  anonymous access rejected (401/403)")
elif len(feats) == 0:
    print("  ✓  anonymous returned 0 items (privacy enforced)")
else:
    print(f"  ✗  PRIVACY LEAK: anonymous got {len(feats)} items!")
    for f in feats[:2]:
        print(f"    {f.get('id')}")
anon.close()


### B.6 — STAC error surfacing: malformed item → proper error code

In [ ]:
r = client.post(
    f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
    content=json.dumps({
        "type": "Feature", "stac_version": "1.0.0",
        "id": f"bad-{RUN_ID}",
        "geometry": "NOT_VALID",
        "bbox": [0],
        "properties": {},
        "links": [], "assets": {},
    }),
)
print(f"POST malformed item: {r.status_code}")
if r.status_code == 500:
    print("  ✗  generic 500 — error masking fix NOT deployed")
elif r.status_code in (400, 422):
    print(f"  ✓  validation error surfaced: {r.status_code}")
    try:
        print(f"  detail: {json.dumps(r.json(), indent=2)[:400]}")
    except Exception:
        print(f"  raw: {r.text[:300]}")
else:
    print(f"  {r.status_code} — {r.text[:200]}")


---
## Teardown

In [ ]:
for cat_id in [PUB_CATALOG_ID, PRIV_CATALOG_ID]:
    r = client.delete(f"/stac/catalogs/{cat_id}", params={"force": "true"})
    print(f"DELETE {cat_id!r}: {r.status_code}")
client.close()
print("Done.")
